Import thư viện & Cấu hình

In [1]:
import tensorflow as tf
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.model_selection import KFold
import tensorflow.keras.backend as K

# --- CẤU HÌNH ---
IMG_WIDTH, IMG_HEIGHT = 224, 224
BATCH_SIZE = 32
EPOCHS = 20        # Custom CNN nhẹ hơn nên có thể tăng Epochs lên 20-30
NUM_FOLDS = 5      # Số vòng lặp K-Fold
LEARNING_RATE = 0.001 # Tốc độ học (Custom CNN thường dùng 0.001, VGG hay dùng 0.0001)

# --- TỰ ĐỘNG PHÁT HIỆN MÔI TRƯỜNG ---
try:
    from google.colab import drive
    print("Detected: GOOGLE COLAB environment")
    drive.mount('/content/drive')
    # SỬA ĐƯỜNG DẪN NÀY THEO FOLDER TRÊN DRIVE CỦA BẠN
    DATA_DIR = '/content/drive/MyDrive/AI_VGG16_Classifier/data/raw' 
except ImportError:
    print("Detected: LOCAL environment")
    DATA_DIR = '../data/raw' 

print(f"Đang tìm dữ liệu tại: {DATA_DIR}")

Detected: LOCAL environment
Đang tìm dữ liệu tại: ../data/raw


Tải danh sách ảnh vào DataFrame

In [2]:
def load_image_paths(data_dir):
    image_dir = Path(data_dir)
    # Tìm tất cả các đuôi ảnh phổ biến
    filepaths = list(image_dir.glob(r'**/*.jpg')) + list(image_dir.glob(r'**/*.png')) + list(image_dir.glob(r'**/*.jpeg'))
    
    # Lấy tên thư mục cha làm nhãn (Label)
    labels = [os.path.split(os.path.split(filepath)[0])[1] for filepath in filepaths]

    filepaths = pd.Series(filepaths, name='Filepath').astype(str)
    labels = pd.Series(labels, name='Label')

    df = pd.concat([filepaths, labels], axis=1)
    
    # Trộn ngẫu nhiên (Shuffle)
    df = df.sample(frac=1).reset_index(drop=True)
    return df

# Chạy hàm tải dữ liệu
try:
    df = load_image_paths(DATA_DIR)
    print(f"Tổng số ảnh tìm thấy: {len(df)}")
    print("\nSố lượng ảnh mỗi lớp:")
    print(df['Label'].value_counts())
except Exception as e:
    print(f"Lỗi tìm ảnh: {e}")

Tổng số ảnh tìm thấy: 267

Số lượng ảnh mỗi lớp:
Label
pins_Messi       102
pins_Benzenma    102
pins_Ronaldo      63
Name: count, dtype: int64


Xây dựng kiến trúc Custom CNN

In [3]:
def build_custom_cnn(num_classes):
    model = Sequential()
    
    # --- BLOCK 1 ---
    # Input nhận ảnh 224x224x3
    model.add(Input(shape=(IMG_WIDTH, IMG_HEIGHT, 3)))
    # Conv2D: Trích xuất đặc trưng
    model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
    # MaxPooling: Giảm kích thước ảnh đi một nửa
    model.add(MaxPooling2D((2, 2)))
    
    # --- BLOCK 2 ---
    model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D((2, 2)))
    
    # --- BLOCK 3 ---
    model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D((2, 2)))
    
    # --- PHẦN PHÂN LOẠI (CLASSIFIER) ---
    model.add(Flatten()) # Duỗi phẳng dữ liệu
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.5)) # Tắt ngẫu nhiên 50% neuron để chống học vẹt (Overfitting)
    
    # Lớp đầu ra (Output Layer)
    model.add(Dense(num_classes, activation='softmax'))
    
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    
    return model

print("Đã khởi tạo hàm build_custom_cnn thành công!")

Đã khởi tạo hàm build_custom_cnn thành công!


Huấn luyện K-Fold

In [4]:
# Khởi tạo K-Fold
kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)

# --- CẤU HÌNH DATA GENERATOR (Quan trọng: Rescale 1./255) ---
train_datagen = ImageDataGenerator(
    rescale=1./255,         # Chuẩn hóa pixel về [0, 1]
    rotation_range=20,      # Xoay ảnh ngẫu nhiên
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    rescale=1./255          # Chỉ chuẩn hóa, không xoay lật
)

acc_per_fold = []
loss_per_fold = []
fold_no = 1

# BẮT ĐẦU VÒNG LẶP
for train_index, val_index in kf.split(df):
    print(f"\nTraining Custom CNN for Fold {fold_no} ...")
    
    train_data = df.iloc[train_index]
    val_data = df.iloc[val_index]
    
    # 1. Tạo Train Generator
    train_gen = train_datagen.flow_from_dataframe(
        train_data, x_col='Filepath', y_col='Label',
        target_size=(IMG_WIDTH, IMG_HEIGHT),
        class_mode='categorical',
        batch_size=BATCH_SIZE,
        shuffle=True
    )
    
    # Lấy danh sách lớp đầy đủ để tránh lỗi thiếu class
    full_classes = list(train_gen.class_indices.keys())
    
    # 2. Tạo Val Generator (Ép buộc dùng full_classes)
    val_gen = val_datagen.flow_from_dataframe(
        val_data, x_col='Filepath', y_col='Label',
        target_size=(IMG_WIDTH, IMG_HEIGHT),
        class_mode='categorical',
        batch_size=BATCH_SIZE,
        shuffle=False,
        classes=full_classes 
    )
    
    # 3. Gọi hàm xây dựng Custom CNN
    num_classes = len(full_classes)
    model = build_custom_cnn(num_classes)
    
    # 4. Callbacks (Lưu file tên khác để không đè lên model VGG cũ)
    checkpoint_path = f"../models/custom_cnn_fold_{fold_no}.h5"
    if 'google.colab' in str(get_ipython()):
         checkpoint_path = f"/content/drive/MyDrive/models/custom_cnn_fold_{fold_no}.h5"

    callbacks = [
        ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    ]
    
    # 5. Training
    try:
        history = model.fit(
            train_gen,
            epochs=EPOCHS, 
            validation_data=val_gen,
            callbacks=callbacks
        )
        
        # Ghi nhận kết quả
        scores = model.evaluate(val_gen, verbose=0)
        print(f'-> Kết quả Fold {fold_no}: Accuracy = {scores[1]*100:.2f}%')
        acc_per_fold.append(scores[1] * 100)
        loss_per_fold.append(scores[0])
        
    except Exception as e:
        print(f"Lỗi tại Fold {fold_no}: {e}")

    # Dọn dẹp RAM
    K.clear_session()
    fold_no += 1

# Báo cáo cuối cùng
print("\n" + "="*30)
if len(acc_per_fold) > 0:
    print(f"TRUNG BÌNH CỘNG (Custom CNN): {np.mean(acc_per_fold):.2f}%")
else:
    print("Chưa chạy xong fold nào.")
print("="*30)


Training Custom CNN for Fold 1 ...
Found 213 validated image filenames belonging to 3 classes.
Found 54 validated image filenames belonging to 3 classes.
Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.3707 - loss: 1.8697
Epoch 1: val_accuracy improved from None to 0.40741, saving model to ../models/custom_cnn_fold_1.h5


7/7 ━━━━━━━━━━━━━━━━━━━━ 19s 2s/step - accuracy: 0.3803 - loss: 1.6379 - val_accuracy: 0.4074 - val_loss: 1.1018
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4455 - loss: 1.0828
Epoch 2: val_accuracy improved from 0.40741 to 0.72222, saving model to ../models/custom_cnn_fold_1.h5


7/7 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step - accuracy: 0.5070 - loss: 1.0572 - val_accuracy: 0.7222 - val_loss: 1.0570
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6333 - loss: 0.9726
Epoch 3: val_accuracy did not improve from 0.72222
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.6244 - loss: 0.9264 - val_accuracy: 0.7222 - val_loss: 0.6986
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6512 - loss: 0.8286
Epoch 4: val_accuracy improved from 0.72222 to 0.96296, saving model to ../models/custom_cnn_fold_1.h5


7/7 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.6808 - loss: 0.7483 - val_accuracy: 0.9630 - val_loss: 0.3167
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7789 - loss: 0.5850
Epoch 5: val_accuracy improved from 0.96296 to 1.00000, saving model to ../models/custom_cnn_fold_1.h5


7/7 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step - accuracy: 0.7981 - loss: 0.5224 - val_accuracy: 1.0000 - val_loss: 0.1658
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8871 - loss: 0.3670
Epoch 6: val_accuracy did not improve from 1.00000
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.8920 - loss: 0.3454 - val_accuracy: 0.9815 - val_loss: 0.1178
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8724 - loss: 0.3589
Epoch 7: val_accuracy did not improve from 1.00000
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.9108 - loss: 0.2624 - val_accuracy: 0.9815 - val_loss: 0.0846
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9380 - loss: 0.2568
Epoch 8: val_accuracy did not improve from 1.00000
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.9531 - loss: 0.1877 - val_accuracy: 1.0000 - val_loss: 0.0533
Epoch 9/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9039 - loss: 0.2537
Epoch 9: val_accuracy did not improve from 1.00000
7/7 ━━━━━━━━━━━━━━━━

...

7/7 ━━━━━━━━━━━━━━━━━━━━ 13s 2s/step - accuracy: 0.3084 - loss: 2.1384 - val_accuracy: 0.6226 - val_loss: 1.0622
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4327 - loss: 1.0835
Epoch 2: val_accuracy did not improve from 0.62264
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.4720 - loss: 1.0758 - val_accuracy: 0.4340 - val_loss: 0.9571
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5290 - loss: 1.0004
Epoch 3: val_accuracy improved from 0.62264 to 0.69811, saving model to ../models/custom_cnn_fold_5.h5


7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 2s/step - accuracy: 0.5701 - loss: 0.9840 - val_accuracy: 0.6981 - val_loss: 0.7439
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6611 - loss: 0.8143
Epoch 4: val_accuracy improved from 0.69811 to 0.81132, saving model to ../models/custom_cnn_fold_5.h5


7/7 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.6776 - loss: 0.7832 - val_accuracy: 0.8113 - val_loss: 0.4673
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6977 - loss: 0.7170
Epoch 5: val_accuracy improved from 0.81132 to 0.98113, saving model to ../models/custom_cnn_fold_5.h5


7/7 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step - accuracy: 0.7477 - loss: 0.6388 - val_accuracy: 0.9811 - val_loss: 0.2358
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8306 - loss: 0.5212
Epoch 6: val_accuracy did not improve from 0.98113
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.8364 - loss: 0.4861 - val_accuracy: 0.9057 - val_loss: 0.2288
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8815 - loss: 0.3869
Epoch 7: val_accuracy improved from 0.98113 to 1.00000, saving model to ../models/custom_cnn_fold_5.h5


7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.8879 - loss: 0.3555 - val_accuracy: 1.0000 - val_loss: 0.0617
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9492 - loss: 0.1934
Epoch 8: val_accuracy did not improve from 1.00000
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.9533 - loss: 0.1674 - val_accuracy: 1.0000 - val_loss: 0.0146
Epoch 9/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8701 - loss: 0.2713
Epoch 9: val_accuracy did not improve from 1.00000
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.9019 - loss: 0.2716 - val_accuracy: 1.0000 - val_loss: 0.0137
Epoch 10/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8532 - loss: 0.4607
Epoch 10: val_accuracy did not improve from 1.00000
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.7991 - loss: 0.5917 - val_accuracy: 1.0000 - val_loss: 0.0541
Epoch 11/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9129 - loss: 0.2417
Epoch 11: val_accuracy did not improve from 1.00000
7/7 ━━━━━━━━━━━━